# Semana 1 · Sistemas de software con LLMs

## Experimento del estudiante

**Pregunta central:** ¿qué cambia arquitectónicamente cuando una parte del software deja de ser determinista?

Este notebook usa GPT-2, un modelo pequeño, abierto y sin instruction tuning. Sirve para ver el mecanismo sin adornos. Todo corre en Google Colab con CPU; no se necesita GPU, llave ni créditos.

Qué se mide hoy: **varianza entre corridas** y **latencia**. Lo demás se observa.


Cómo trabajar este notebook: cada bloque tiene una celda **Predicción** que se llena antes de ejecutar, y preguntas al final. Los `TODO` marcan lo que hay que completar. El notebook corre de principio a fin aunque los TODO estén vacíos; llenarlos es la parte que se evalúa.

## 0. Preparación

In [ ]:
# En Colab estas librerías ya vienen instaladas. La línea queda por si se ejecuta en otro entorno.
# !pip install -q transformers torch

In [ ]:
import os, time, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "gpt2"  # 124 millones de parámetros, publicado en 2019
tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32)
modelo.eval()
torch.manual_seed(0)
print("Modelo cargado. Parámetros:", f"{sum(p.numel() for p in modelo.parameters())/1e6:.0f} M")

Una sola función para llamar al modelo. Recibe texto y devuelve texto. Por dentro hace lo que se explica en la semana 2; hoy es una caja.

`do_sample=False` elige siempre el token más probable (equivale a temperatura 0). `do_sample=True` muestrea según las probabilidades y la temperatura.

In [ ]:
def generar(prompt: str, max_new_tokens: int = 12, do_sample: bool = True, temperature: float = 1.0) -> str:
    """Llama a GPT-2 y devuelve solo el texto nuevo (sin el prompt)."""
    ids = tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        salida = modelo.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_k=None if do_sample else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(salida[0, ids.shape[1]:], skip_special_tokens=True)

print(repr(generar("The capital of France is", do_sample=False)))

## 1. Dos contratos: una función y un modelo

La tarea es contar palabras. La primera implementación es una función. La segunda le pide al modelo que lo haga.

In [ ]:
texto = "El sistema debe validar el RFC antes de emitir la factura y registrar cada intento"

def contar_palabras(texto: str) -> int:
    return len(texto.split())

prompt = f"Text: {texto}\nNumber of words in the text:"
print("Palabras reales:", contar_palabras(texto))

**Predicción (antes de ejecutar la siguiente celda).** Si se llama diez veces a la función y diez veces al modelo con el mismo texto:

* ¿Cuántas salidas distintas produce la función? ¿Cuántas el modelo?
* ¿Cuánto tarda cada llamada, en orden de magnitud?

In [ ]:
# TODO: escribe tus predicciones antes de ejecutar el bloque.
PREDICCION_SALIDAS_DISTINTAS_MODELO = None   # un entero entre 1 y 10
PREDICCION_LATENCIA_MODELO_S = None          # segundos por llamada, aproximado

In [ ]:
print("Función determinista")
for _ in range(10):
    t = time.perf_counter()
    n = contar_palabras(texto)
    dt = time.perf_counter() - t
    print(f"  {n}   {dt*1e6:.0f} µs")

print("\nModelo de lenguaje (muestreo activado)")
salidas, latencias = [], []
for _ in range(10):
    t = time.perf_counter()
    s = generar(prompt, max_new_tokens=6, do_sample=True)
    dt = time.perf_counter() - t
    salidas.append(s); latencias.append(dt)
    print(f"  {s!r:45}   {dt:.2f} s")

print(f"\nSalidas distintas del modelo: {len(set(salidas))} de 10")
print(f"Latencia del modelo: min {min(latencias):.2f} s, max {max(latencias):.2f} s")

**Preguntas del bloque 1.**

1. ¿Cuántas salidas distintas produjo el modelo? ¿Coincidió con tu predicción?
2. ¿Cuántas de las diez salidas contienen el número correcto de palabras?
3. ¿Cuál es la diferencia de latencia entre la función y el modelo, en órdenes de magnitud?
4. Si otro programa consumiera la salida del modelo, ¿qué tendría que hacer antes de usarla?

In [ ]:
# TODO: responde en texto.
RESPUESTAS_BLOQUE_1 = """
1.
2.
3.
4.
"""

## 2. Temperatura

`do_sample=False` (temperatura 0) elige siempre el token más probable. Con muestreo, la temperatura controla cuánto se aplana la distribución antes de elegir. La semana 2 explica el mecanismo; hoy se observa el efecto.

**Predicción.** Con temperatura 0, ¿las tres salidas serán iguales? Con temperatura 1.5, ¿serán más o menos coherentes que con 0.7?

In [ ]:
# TODO
PREDICCION_T0_IGUALES = None        # True / False
PREDICCION_T15_MAS_COHERENTE = None # True / False

In [ ]:
prompt2 = "The software architect reviewed the design and concluded that"

print("Temperatura 0 (do_sample=False), tres llamadas")
for _ in range(3):
    print("  ", repr(generar(prompt2, do_sample=False)))

for temp in (0.7, 1.5):
    print(f"\nTemperatura {temp}, tres llamadas")
    for _ in range(3):
        print("  ", repr(generar(prompt2, do_sample=True, temperature=temp)))

**Preguntas del bloque 2.**

1. ¿Temperatura 0 produjo tres salidas idénticas? ¿Eso convierte al sistema en determinista? Argumenta.
2. ¿Qué pasó con la coherencia al subir la temperatura a 1.5?
3. Si tu aplicación clasifica correos, ¿qué temperatura usarías y por qué? Si genera ideas para un nombre de producto, ¿cambiaría?

In [ ]:
# TODO
RESPUESTAS_BLOQUE_2 = """
1.
2.
3.
"""

## 3. La alucinación

Se le piden al modelo datos verificables. GPT-2 se entrenó principalmente con texto en inglés, así que los prompts van en inglés para ver la fluidez con la que responde. Después se le da una instrucción en español.

In [ ]:
prompts = [
    "The capital of Australia is the city of",
    "Gabriel García Márquez was born in the year",
    "Today's date is",
    "The result of 17 times 23 is",
]
for p in prompts:
    print(f"{p!r}\n   -> {generar(p, max_new_tokens=8, do_sample=False)!r}\n")

In [ ]:
# Una instrucción explícita, en español. El modelo no fue entrenado para obedecer instrucciones.
instruccion = "Responde únicamente con la palabra SÍ o NO. ¿Es 7 un número primo? Respuesta:"
for _ in range(3):
    print(repr(generar(instruccion, max_new_tokens=8, do_sample=True)))

**Preguntas del bloque 3.**

1. ¿Qué capital dio el modelo para Australia? ¿Es correcta? ¿Cómo lo sabrías sin buscarlo?
2. ¿Qué dio para 17 × 23? Compáralo con `17 * 23` en Python. ¿Qué componente debería hacer esta operación en una aplicación real?
3. ¿Obedeció la instrucción "responde SÍ o NO"? Propón una explicación con lo que sabes hasta ahora (la semana 3 la completa).
4. Escribe tu propia definición de alucinación en términos del ciclo de generación.

In [ ]:
# TODO
RESPUESTAS_BLOQUE_3 = """
1.
2.
3.
4.
"""
print("Comprobación:", 17 * 23)

## 4. La arquitectura mínima

Una aplicación que clasifica tickets de soporte en tres categorías. Cuatro responsabilidades, una sola probabilística:

1. Construcción del prompt.
2. Llamada al modelo.
3. Validación determinista de la salida.
4. Registro de cada llamada (observabilidad).

Con GPT-2 la validación va a fallar en muchos casos. Eso es lo que se quiere ver: la falla queda contenida y contada, en lugar de propagarse.

In [ ]:
OPCIONES = ["billing", "access", "bug"]   # categorías cerradas (en inglés por GPT-2)
registro = []                              # observabilidad mínima: una lista en memoria

def construir_prompt(texto: str) -> str:
    # Dos ejemplos en el prompt para que GPT-2 siga el formato. La semana 4 estudia esto a fondo.
    return (
        'Ticket: "I need a refund for the duplicate charge."\nCategory: billing\n\n'
        'Ticket: "The login page returns an error."\nCategory: access\n\n'
        f'Ticket: "{texto}"\nCategory:'
    )

def validar(salida: str, opciones: list[str]):
    """Devuelve la opción encontrada en la salida, o None si no hay exactamente una."""
    encontradas = [o for o in opciones if o in salida.lower()]
    return encontradas[0] if len(encontradas) == 1 else None

def registrar(prompt, salida, etiqueta, latencia):
    registro.append({"prompt": prompt, "salida": salida, "etiqueta": etiqueta, "latencia_s": round(latencia, 3)})

def clasificar(texto: str) -> dict:
    prompt = construir_prompt(texto)                     # 1
    t = time.perf_counter()
    salida = generar(prompt, max_new_tokens=4, do_sample=False)   # 2
    latencia = time.perf_counter() - t
    etiqueta = validar(salida, OPCIONES)                 # 3
    registrar(prompt, salida, etiqueta, latencia)        # 4
    return {"etiqueta": etiqueta, "latencia_s": latencia}

In [ ]:
tickets = [
    "I was charged twice for my July subscription.",
    "I cannot log in, it says my password is wrong.",
    "The export to PDF shows blank charts.",
    "Please send me the invoice for June again.",
    "The app crashes when I open the reports tab.",
]
for tk in tickets:
    r = clasificar(tk)
    print(f"{r['etiqueta']!s:8} {r['latencia_s']:.2f} s   {tk}")

validas = sum(1 for r in registro if r["etiqueta"] is not None)
print(f"\nSalidas válidas: {validas} de {len(registro)}")
print("Latencia media:", f"{sum(r['latencia_s'] for r in registro)/len(registro):.2f} s")
print("\nÚltimas salidas crudas del modelo:")
for r in registro[-5:]:
    print("  ", repr(r["salida"]), "->", r["etiqueta"])

**Preguntas del bloque 4.**

1. ¿Cuántas salidas pasaron la validación? ¿Qué devolvió el modelo en las que fallaron? De las válidas, ¿cuántas son correctas? ¿Es lo mismo cumplir el formato que acertar?
2. Propón qué debería hacer `clasificar()` cuando `validar()` devuelve `None`. Escribe la política en una línea.
3. ¿Qué campo agregarías al registro para poder depurar una queja de un usuario dentro de una semana?

**TODO de código.** Modifica `validar()` para que acepte una salida aunque contenga más de una opción, eligiendo la que aparece primero en el texto. Ejecuta de nuevo la celda de los tickets y compara el número de salidas válidas.

In [ ]:
# TODO: nueva versión de validar(). Deja la original arriba para comparar.
def validar_v2(salida: str, opciones: list[str]):
    # pista: usar salida.lower().find(o) para cada opción y quedarse con la posición mínima distinta de -1
    return validar(salida, opciones)   # reemplazar

registro_v2 = []
for tk in tickets:
    salida = generar(construir_prompt(tk), max_new_tokens=4, do_sample=False)
    registro_v2.append(validar_v2(salida, OPCIONES))
print("Válidas con validar_v2:", sum(1 for e in registro_v2 if e is not None), "de", len(tickets))

RESPUESTAS_BLOQUE_4 = """
1.
2.
3.
"""

## 5. Cierre

Con lo observado, responde en cinco líneas la pregunta central: ¿qué cambia arquitectónicamente cuando una parte del software deja de ser determinista? Usa evidencia de los bloques 1 a 4.

In [ ]:
# TODO
RESPUESTA_PREGUNTA_CENTRAL = """

"""

### Para la práctica 1

Con el diagrama de la arquitectura mínima como plantilla, la práctica 1 pide la arquitectura V0 de la mesa de soporte del curso: qué entra, qué construye el prompt, dónde está el modelo, qué valida la salida, qué se registra, y qué parte del problema se resuelve sin modelo. Instrucciones, starter y rúbrica en `practice/`.

Opcional, sin peso: repetir el diagrama sobre un caso propio.